# Supply Chain Service Level Tracking Analysis

In [1]:
# Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Cleaning & Exploration

In [4]:
# Load data
customers = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_customers.csv")
date = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_date.csv")
products = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_products.csv")
targets = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/dim_targets_orders.csv")
orders = pd.read_csv("G:/Other computers/Laptop - GoogleDrive/data analytics/projects/supply chain _ codebasics/analysis_dataset/fact_order_lines.csv")


### Preview data set

In [17]:
customers.head(3)

,customer_id,customer_name,city
0,789201,Rel Fresh,Surat
1,789202,Rel Fresh,Ahmedabad
2,789203,Rel Fresh,Vadodara


In [18]:
date.head(3)

,date,mmm_yy,week_no
0,01-Apr-22,01-Apr-22,W 14
1,03-Apr-22,01-Apr-22,W 15
2,04-Apr-22,01-Apr-22,W 15


In [19]:
products.head(3)

,product_id,product_name,category
0,25891101,AM Milk 500,Dairy
1,25891102,AM Milk 250,Dairy
2,25891103,AM Milk 100,Dairy


In [20]:
targets.head(3)

,customer_id,ontime_target%,infull_target%,otif_target%
0,789201,87,81,70
1,789202,85,81,69
2,789203,92,76,70


In [21]:
orders.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty
0,FMR34203601,"Tuesday, March 1, 2022",789203,25891601,110,"Friday, March 4, 2022","Friday, March 4, 2022",110
1,FMR32320302,"Tuesday, March 1, 2022",789320,25891203,347,"Wednesday, March 2, 2022","Wednesday, March 2, 2022",347
2,FMR33320501,"Tuesday, March 1, 2022",789320,25891203,187,"Thursday, March 3, 2022","Thursday, March 3, 2022",150


### Standardise column names for all tables

In [23]:
# Standardise column names
for df in [customers, date, products, targets, orders]:
    df.columns = df.columns.str.lower().str.strip()

### Validate **`customers`** table
- data types
- missing values
- duplicates

In [27]:
customers.dtypes

customer_id       int64
customer_name    object
city             object
dtype: object

In [30]:
customers.isnull().sum()

customer_id      0
customer_name    0
city             0
dtype: int64

In [31]:
customers.duplicated().sum()

np.int64(0)

In [39]:
customers.duplicated(subset=['customer_id', 'customer_name', 'city']).sum()

np.int64(0)

### Validate **`date`** table
- data types
- missing values
- duplicates

In [41]:
date.dtypes

date       object
mmm_yy     object
week_no    object
dtype: object

#### parse datetime data type for columns:
- date
- mmm-yy

In [47]:
# Convert date column
date['date'] = pd.to_datetime(date['date'], format='%d-%b-%y')

# Convert mmm_yy to datetime (month-level)
date['mmm_yy'] = pd.to_datetime(date['mmm_yy'], format='%d-%b-%y')

# week_no stays as text
date['week_no'] = date['week_no'].astype('string')

date.dtypes

date       datetime64[ns]
mmm_yy     datetime64[ns]
week_no    string[python]
dtype: object

In [48]:
date.head(3)

,date,mmm_yy,week_no
0,2022-04-01,2022-04-01,W 14
1,2022-04-03,2022-04-01,W 15
2,2022-04-04,2022-04-01,W 15


In [42]:
date.isnull().sum()

date       0
mmm_yy     0
week_no    0
dtype: int64

In [43]:
date.duplicated().sum()

np.int64(0)

### Validate **`products`** table
- data types
- missing values
- duplicates

In [49]:
products.dtypes

product_id       int64
product_name    object
category        object
dtype: object

In [50]:
products.isnull().sum()

product_id      0
product_name    0
category        0
dtype: int64

In [52]:
products.duplicated(subset=['product_id', 'product_name']).sum()

np.int64(0)

### Validate **`targets`** table
- data types
- missing values
- duplicates

In [53]:
targets.dtypes

customer_id       int64
ontime_target%    int64
infull_target%    int64
otif_target%      int64
dtype: object

In [54]:
targets.isnull().sum()

customer_id       0
ontime_target%    0
infull_target%    0
otif_target%      0
dtype: int64

In [56]:
targets.duplicated(subset=['customer_id']).sum()

np.int64(0)

### Validate **`orders`** table
- data types
- missing values
- duplicates

In [58]:
orders.dtypes

order_id                object
order_placement_date    object
customer_id              int64
product_id               int64
order_qty                int64
agreed_delivery_date    object
actual_delivery_date    object
delivery_qty             int64
dtype: object

#### parse datetime data type for columns:
- order_placement_date
- agreed_delivery_date
- actual_delivery_date

In [62]:
orders['order_placement_date'] = pd.to_datetime(orders['order_placement_date'])
orders['agreed_delivery_date'] = pd.to_datetime(orders['agreed_delivery_date'])
orders['actual_delivery_date'] = pd.to_datetime(orders['actual_delivery_date'])

orders.dtypes

order_id                        object
order_placement_date    datetime64[ns]
customer_id                      int64
product_id                       int64
order_qty                        int64
agreed_delivery_date    datetime64[ns]
actual_delivery_date    datetime64[ns]
delivery_qty                     int64
dtype: object

In [63]:
orders.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150


In [65]:
orders.isnull().sum()

order_id                0
order_placement_date    0
customer_id             0
product_id              0
order_qty               0
agreed_delivery_date    0
actual_delivery_date    0
delivery_qty            0
dtype: int64

In [66]:
orders.duplicated().sum()

np.int64(0)

### Merge  **`customers`** & **`products`** with **`orders`**

In [67]:
orders = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products, on='product_id', how='left')
)

orders.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,customer_name,city,product_name,category
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110,Rel Fresh,Vadodara,AM Tea 500,beverages
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347,Chiptec Stores,Surat,AM Butter 500,Dairy
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150,Chiptec Stores,Surat,AM Butter 500,Dairy


In [68]:
orders.shape

(57096, 12)

### Create Core Service Level Agreement (SLA) Flags

In [75]:
# On-Time Delivery
orders['on_time_flag'] = (
    orders['actual_delivery_date'] 
    <= orders['agreed_delivery_date']
)

# In-Full Delivery
orders['in_full_flag'] = (
    orders['delivery_qty'] 
    >= orders['order_qty']
)

# OTIF (On-Time + In-Full)
orders['otif_flag'] = (
    orders['on_time_flag'] & orders['in_full_flag']
)

orders.head(3)

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,customer_name,city,product_name,category,on_time_flag,in_full_flag,otif_flag
0,FMR34203601,2022-03-01,789203,25891601,110,2022-03-04,2022-03-04,110,Rel Fresh,Vadodara,AM Tea 500,beverages,True,True,True
1,FMR32320302,2022-03-01,789320,25891203,347,2022-03-02,2022-03-02,347,Chiptec Stores,Surat,AM Butter 500,Dairy,True,True,True
2,FMR33320501,2022-03-01,789320,25891203,187,2022-03-03,2022-03-03,150,Chiptec Stores,Surat,AM Butter 500,Dairy,True,False,False
